In [1]:
## imports for XGBoost multi-class classification
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import os

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)

np.random.seed(42)

## loading preprocessed data (from notebook 02) + reconstruction errors (from notebook 03)
X_train = np.load('../data/processed/X_train.npy')
X_test  = np.load('../data/processed/X_test.npy')
y_train = np.load('../data/processed/y_train.npy')
y_test  = np.load('../data/processed/y_test.npy')

recon_error_train = np.load('../data/processed/recon_error_train.npy')
recon_error_test  = np.load('../data/processed/recon_error_test.npy')

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"recon_error_train: {recon_error_train.shape}, recon_error_test: {recon_error_test.shape}")

label_map = {0: 'Normal', 1: 'DoS', 2: 'Probe', 3: 'R2L', 4: 'U2R'}
print("\nClass distribution (train):")
for cid, cname in label_map.items():
    print(f"  {cname:8s}: {(y_train == cid).sum()}")

X_train: (125973, 122), X_test: (22251, 122)
recon_error_train: (125973,), recon_error_test: (22251,)

Class distribution (train):
  Normal  : 67343
  DoS     : 45927
  Probe   : 11656
  R2L     : 995
  U2R     : 52


In [2]:
## building both feature sets for the ablation study
# Version A — baseline: 122 features only, no autoencoder signal
X_train_A = X_train
X_test_A  = X_test

# Version B — hybrid: 123 features, recon error injected as the last column
# reshape recon_error from (n,) to (n,1) so it can be hstacked as a column
X_train_B = np.hstack([X_train, recon_error_train.reshape(-1, 1)])
X_test_B  = np.hstack([X_test, recon_error_test.reshape(-1, 1)])

print(f"Version A (baseline) — X_train_A: {X_train_A.shape}, X_test_A: {X_test_A.shape}")
print(f"Version B (hybrid)   — X_train_B: {X_train_B.shape}, X_test_B: {X_test_B.shape}")

# sanity check: last column of Version B should exactly equal recon_error_train
assert np.allclose(X_train_B[:, -1], recon_error_train), "Injection mismatch!"
print("\nSanity check passed: last column of X_train_B matches recon_error_train")

Version A (baseline) — X_train_A: (125973, 122), X_test_A: (22251, 122)
Version B (hybrid)   — X_train_B: (125973, 123), X_test_B: (22251, 123)

Sanity check passed: last column of X_train_B matches recon_error_train


In [4]:
# only needed if error_summary_by_class isn't already in memory
def error_summary_by_class(recon_error, y, dataset_name):
    label_map = {0: 'Normal', 1: 'DoS', 2: 'Probe', 3: 'R2L', 4: 'U2R'}
    print(f"--- {dataset_name} ---")
    for class_id, class_name in label_map.items():
        mask = (y == class_id)
        errs = recon_error[mask]
        print(f"{class_name:8s} (n={mask.sum():6d}): mean={errs.mean():.6f}, median={np.median(errs):.6f}")
    print()

# the content-only arrays you already saved
recon_error_content_train = np.load('../data/processed/recon_error_content_train.npy')
recon_error_content_test  = np.load('../data/processed/recon_error_content_test.npy')

error_summary_by_class(recon_error_content_train, y_train, "TRAIN — content-only error")
error_summary_by_class(recon_error_content_test,  y_test,  "TEST — content-only error")

# quick side-by-side: does content-only separate R2L/U2R from Normal
# better than the full-feature aggregate did?
recon_error_train = np.load('../data/processed/recon_error_train.npy')
recon_error_test  = np.load('../data/processed/recon_error_test.npy')

for cls in [3, 4]:  # R2L, U2R
    name = {3: 'R2L', 4: 'U2R'}[cls]
    normal_mean_full    = recon_error_test[y_test == 0].mean()
    normal_mean_content = recon_error_content_test[y_test == 0].mean()
    cls_mean_full    = recon_error_test[y_test == cls].mean()
    cls_mean_content = recon_error_content_test[y_test == cls].mean()
    print(f"{name}: full-feature ratio={cls_mean_full/normal_mean_full:.2f}x, "
          f"content-only ratio={cls_mean_content/normal_mean_content:.2f}x")

--- TRAIN — content-only error ---
Normal   (n= 67343): mean=0.000393, median=0.000000
DoS      (n= 45927): mean=0.000231, median=0.000000
Probe    (n= 11656): mean=0.000632, median=0.000000
R2L      (n=   995): mean=0.004757, median=0.000000
U2R      (n=    52): mean=0.044352, median=0.076936

--- TEST — content-only error ---
Normal   (n=  9711): mean=0.000274, median=0.000000
DoS      (n=  7167): mean=0.003763, median=0.000000
Probe    (n=  2421): mean=0.002717, median=0.000000
R2L      (n=  2885): mean=0.008992, median=0.000000
U2R      (n=    67): mean=0.053800, median=0.019909

R2L: full-feature ratio=5.02x, content-only ratio=32.80x
U2R: full-feature ratio=5.05x, content-only ratio=196.28x


In [6]:
recon_error_other_train = np.load('../data/processed/recon_error_other_train.npy')
recon_error_other_test  = np.load('../data/processed/recon_error_other_test.npy')

error_summary_by_class(recon_error_other_train, y_train, "TRAIN — non-content (traffic+basic) error")
error_summary_by_class(recon_error_other_test,  y_test,  "TEST — non-content (traffic+basic) error")

for cls in [1, 2]:  # DoS, Probe
    name = {1: 'DoS', 2: 'Probe'}[cls]
    normal_mean = recon_error_other_test[y_test == 0].mean()
    cls_mean    = recon_error_other_test[y_test == cls].mean()
    print(f"{name}: non-content ratio={cls_mean/normal_mean:.2f}x")

--- TRAIN — non-content (traffic+basic) error ---
Normal   (n= 67343): mean=0.001195, median=0.000038
DoS      (n= 45927): mean=0.040297, median=0.039677
Probe    (n= 11656): mean=0.027984, median=0.023851
R2L      (n=   995): mean=0.003192, median=0.000357
U2R      (n=    52): mean=0.003649, median=0.000760

--- TEST — non-content (traffic+basic) error ---
Normal   (n=  9711): mean=0.001850, median=0.000030
DoS      (n=  7167): mean=0.031844, median=0.034874
Probe    (n=  2421): mean=0.038572, median=0.042567
R2L      (n=  2885): mean=0.008368, median=0.001847
U2R      (n=    67): mean=0.003086, median=0.001105

DoS: non-content ratio=17.22x
Probe: non-content ratio=20.85x


In [7]:
## computing sample weights to counter severe class imbalance
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

print("Sample weight per class (should be higher for rarer classes):")
for cid, cname in label_map.items():
    w = sample_weights[y_train == cid][0]  # weight is identical within a class
    print(f"  {cname:8s} (n={int((y_train==cid).sum()):6d}): weight = {w:.4f}")

print(f"\nsample_weights shape: {sample_weights.shape}")
print(f"Ratio of U2R weight to Normal weight: {sample_weights[y_train==4][0] / sample_weights[y_train==0][0]:.1f}x")

Sample weight per class (should be higher for rarer classes):
  Normal   (n= 67343): weight = 0.3741
  DoS      (n= 45927): weight = 0.5486
  Probe    (n= 11656): weight = 2.1615
  R2L      (n=   995): weight = 25.3212
  U2R      (n=    52): weight = 484.5115

sample_weights shape: (125973,)
Ratio of U2R weight to Normal weight: 1295.1x


In [8]:
## Version A — baseline XGBoost, 122 features (no reconstruction error)
xgb_A = XGBClassifier(
    objective='multi:softmax',
    num_class=5,
    eval_metric='mlogloss',
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_A.fit(X_train_A, y_train, sample_weight=sample_weights)

## evaluate on test set
y_pred_A = xgb_A.predict(X_test_A)
f1_macro_A = f1_score(y_test, y_pred_A, average='macro')

print(f"Version A (baseline, 122 features) — Macro F1: {f1_macro_A:.4f}\n")
print(classification_report(y_test, y_pred_A, target_names=list(label_map.values())))

Version A (baseline, 122 features) — Macro F1: 0.6043

              precision    recall  f1-score   support

      Normal       0.71      0.97      0.82      9711
         DoS       0.97      0.87      0.91      7167
       Probe       0.81      0.73      0.77      2421
         R2L       0.97      0.09      0.16      2885
         U2R       0.70      0.24      0.36        67

    accuracy                           0.80     22251
   macro avg       0.83      0.58      0.60     22251
weighted avg       0.84      0.80      0.76     22251



In [9]:
recon_error_content_train = np.load('../data/raw/recon_error_content_train.npy')
recon_error_content_test  = np.load('../data/raw/recon_error_content_test.npy')

X_train_hybrid = np.hstack([
    X_train,
    recon_error_content_train.reshape(-1, 1),
    recon_error_other_train.reshape(-1, 1),
])
X_test_hybrid = np.hstack([
    X_test,
    recon_error_content_test.reshape(-1, 1),
    recon_error_other_test.reshape(-1, 1),
])
print(f"Hybrid shape: {X_train_hybrid.shape}")   # (125973, 124)
print(f"Baseline shape: {X_train.shape}")         # (125973, 122)

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

xgb_hybrid = XGBClassifier(
    objective='multi:softprob', num_class=5,
    n_estimators=200, max_depth=8, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, n_jobs=-1
)
xgb_hybrid.fit(X_train_hybrid, y_train, sample_weight=sample_weights)

y_pred_hybrid = xgb_hybrid.predict(X_test_hybrid)
print(f"\nHybrid (content+non-content) macro F1: {f1_score(y_test, y_pred_hybrid, average='macro'):.4f}\n")
print(classification_report(y_test, y_pred_hybrid, target_names=list(label_map.values())))

Hybrid shape: (125973, 124)
Baseline shape: (125973, 122)

Hybrid (content+non-content) macro F1: 0.6386

              precision    recall  f1-score   support

      Normal       0.74      0.97      0.84      9711
         DoS       0.97      0.89      0.93      7167
       Probe       0.83      0.82      0.82      2421
         R2L       0.98      0.13      0.24      2885
         U2R       0.58      0.27      0.37        67

    accuracy                           0.82     22251
   macro avg       0.82      0.62      0.64     22251
weighted avg       0.85      0.82      0.79     22251



In [10]:
def bootstrap_per_class_diff(y_true, y_pred_baseline, y_pred_hybrid,
                              label_map, n_bootstrap=1000, random_state=42):
    rng = np.random.RandomState(random_state)
    n = len(y_true)
    y_true = np.asarray(y_true)
    y_pred_baseline = np.asarray(y_pred_baseline)
    y_pred_hybrid = np.asarray(y_pred_hybrid)

    class_ids = list(label_map.keys())
    diffs = {cid: np.empty(n_bootstrap) for cid in class_ids}
    macro_diffs = np.empty(n_bootstrap)

    for i in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        f1_base   = f1_score(y_true[idx], y_pred_baseline[idx], average=None, labels=class_ids, zero_division=0)
        f1_hybrid = f1_score(y_true[idx], y_pred_hybrid[idx],   average=None, labels=class_ids, zero_division=0)
        for j, cid in enumerate(class_ids):
            diffs[cid][i] = f1_hybrid[j] - f1_base[j]
        macro_diffs[i] = f1_hybrid.mean() - f1_base.mean()

    print("Per-class F1 difference (hybrid - baseline), 95% CI:\n")
    for cid in class_ids:
        d = diffs[cid]
        lo, hi = np.percentile(d, [2.5, 97.5])
        verdict = "significant" if (lo > 0 or hi < 0) else "within noise"
        print(f"  {label_map[cid]:8s}: mean={d.mean():+.4f}  CI=[{lo:+.4f}, {hi:+.4f}]  -> {verdict}")

    lo, hi = np.percentile(macro_diffs, [2.5, 97.5])
    verdict = "significant" if (lo > 0 or hi < 0) else "within noise"
    print(f"\n  {'Macro F1':8s}: mean={macro_diffs.mean():+.4f}  CI=[{lo:+.4f}, {hi:+.4f}]  -> {verdict}")

    return diffs, macro_diffs

label_map = {0: 'Normal', 1: 'DoS', 2: 'Probe', 3: 'R2L', 4: 'U2R'}
diffs, macro_diffs = bootstrap_per_class_diff(y_test, y_pred_A, y_pred_hybrid, label_map)

Per-class F1 difference (hybrid - baseline), 95% CI:

  Normal  : mean=+0.0194  CI=[+0.0176, +0.0214]  -> significant
  DoS     : mean=+0.0127  CI=[+0.0107, +0.0148]  -> significant
  Probe   : mean=+0.0547  CI=[+0.0458, +0.0630]  -> significant
  R2L     : mean=+0.0731  CI=[+0.0608, +0.0852]  -> significant
  U2R     : mean=+0.0118  CI=[-0.0616, +0.0860]  -> within noise

  Macro F1: mean=+0.0344  CI=[+0.0195, +0.0499]  -> significant


In [11]:
diffs, macro_diffs = bootstrap_per_class_diff(
    y_test, y_pred_A, y_pred_hybrid, label_map, n_bootstrap=5000
)

Per-class F1 difference (hybrid - baseline), 95% CI:

  Normal  : mean=+0.0194  CI=[+0.0175, +0.0212]  -> significant
  DoS     : mean=+0.0127  CI=[+0.0107, +0.0148]  -> significant
  Probe   : mean=+0.0547  CI=[+0.0462, +0.0633]  -> significant
  R2L     : mean=+0.0730  CI=[+0.0607, +0.0858]  -> significant
  U2R     : mean=+0.0119  CI=[-0.0578, +0.0862]  -> within noise

  Macro F1: mean=+0.0343  CI=[+0.0201, +0.0494]  -> significant


## Baseline vs. Hybrid — Percentage Improvement Summary

Hyperparameters (`n_estimators=200, max_depth=8, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8`) were selected via grid search prior to this comparison and held fixed across both models.

| Class     | Baseline F1 | Hybrid F1 | Δ (abs) | % Improvement |
|-----------|:-----------:|:---------:|:-------:|:--------------:|
| Normal    | 0.82        | 0.84      | +0.02   | +2.4%          |
| DoS       | 0.91        | 0.93      | +0.02   | +2.2%          |
| Probe     | 0.77        | 0.82      | +0.05   | +6.5%          |
| R2L       | 0.16        | 0.24      | +0.08   | +50.0%         |
| U2R       | 0.36        | 0.37      | +0.01   | +2.8%          |
| **Macro F1** | **0.6043**  | **0.6386** | **+0.0343** | **+5.7%**      |

Bootstrapped 95% CIs (n=5000) confirm the Normal, DoS, Probe, and R2L gains are statistically significant; U2R remains within noise (CI: [-0.0578, +0.0862]) due to its small test support (n=67).
